# Questão 2 — Extração de argumentos: demonstração

Uma demonstração curta da solução descrita no `questao2.md`, na mesma ordem do texto, usando as 226 sentenças reais do `sentencas_tjsp.json` (Juizado Especial Cível do TJSP, 1º semestre de 2024, pseudonimizadas).

- **Roda de verdade, nas sentenças reais:** tudo o que é código: dividir em parágrafos, separar as partes pelas frases fixas, marcar citações, extrair os fundamentos citados, mascarar nomes, valores, datas e empresas, juntar, decidir e montar o JSON.
- **Montado, mas não enviado:** as perguntas ao Jev. O Jev está em acesso antecipado; o caderno mostra exatamente o que seria enviado. O formato do pedido é ilustrativo.
- **Exemplos escritos à mão, marcados como exemplo:** a taxonomia de negativação, as respostas do Jev e as probabilidades mínimas usadas nas etapas 3 e 4. Servem só para mostrar o caminho até o JSON; não são saída de modelo.
- **Sem métricas:** o caderno mostra o fluxo, não mede qualidade.

**A solução, como no texto:** em vez de um LLM escrever os argumentos, uma lista fechada de tipos de argumento para cada tipo de processo (a taxonomia), e o Jev classifica os parágrafos da fundamentação nessa lista. **Premissas:** a 1 (num mesmo tipo de processo os argumentos dos juízes se repetem, não são infinitos) é o que justifica a lista fechada; a 2 (as três partes e as frases fixas) é conferida na seção 0; a 3 (a plataforma já tem sentenças de cada tipo) é o próprio `sentencas_tjsp.json`; a 4 (o Jev decide: recebe um texto e perguntas fechadas e devolve a probabilidade de cada resposta) é o formato das perguntas nas seções 2 a 4.

| Seção | No `questao2.md` |
|---|---|
| 0 | Premissas: tipo de processo e frases fixas |
| 1 | Montagem da taxonomia (passos 1 a 3), taxonomia geral e lista de pedidos |
| 2 | Extração: preparação da sentença (parágrafos, partes, citações) |
| 3 | Etapa 1: pedidos |
| 4 | Etapa 2: argumentos, e sentenças longas |
| 5 | Etapas 3 e 4: juntar e decidir pela probabilidade mínima |
| 6 | Saída estruturada e regra do JSON |
| 7 | Revisão humana |
| 8 | Qualidade |
| 9 | Versionamento |
| 10 | Argumentos inventados: o caminho com um LLM (plano B) |
| 11 | Cobertura do texto e trade-offs |

In [1]:
import hashlib
import json
import re
from collections import Counter

DADOS = json.load(open("sentencas_tjsp.json"))
SENTENCAS = DADOS["sentencas"]
EXEMPLO = SENTENCAS[61]   # negativação no SCR-BACEN, julgada improcedente


def tipo_de_processo(sentenca):
    """O tipo de processo vem da classe e do assunto da sentença."""
    return (DADOS["classe"], sentenca["assunto"])


print(len(SENTENCAS), "sentenças reais")
print("tipo de processo do exemplo:", tipo_de_processo(EXEMPLO))

226 sentenças reais
tipo de processo do exemplo: ('Procedimento do Juizado Especial Cível', 'Inclusão Indevida em Cadastro de Inadimplentes')


## 0. Premissa 2: as frases fixas

A sentença tem relatório, fundamentação e dispositivo, e os juízes marcam a passagem entre eles com frases fixas ("Fundamento e decido", "Ante o exposto"). O texto chega como texto: em produção o PDF é lido página a página e a página escaneada vai para OCR; aqui as sentenças já vêm em texto, então cada parágrafo tem posição no texto, mas não página nem retângulo no PDF.

In [2]:
FRASE_FUNDAMENTACAO = re.compile(r"(?i)^(vistos\b.*|fundamento( e decido)?\.?|decido\.?|(relatório dispensado|dispensado o relatório).*"
                                 r"|é o (breve )?relat(o|ório).*)$")
FRASE_DISPOSITIVO = re.compile(r"(?i)^(ante o exposto|diante do exposto|em razão do exposto|do exposto|pelo exposto|posto isso"
                               r"|isto posto|por tais razões|diante disso)\b.{0,120}?\b(julg|homolog|extin|conden|declar|acolh|rejeit)")


def paragrafos(texto):
    """Divide o texto em parágrafos numerados, com a posição de cada um no texto."""
    lista, pos = [], 0
    for linha in texto.split("\n"):
        ini = texto.index(linha, pos)
        pos = ini + len(linha)
        if linha.strip():
            lista.append({"n": len(lista) + 1, "char_ini": ini, "char_fim": pos, "texto": linha.strip()})
    return lista


def separar_partes(pars):
    """Relatório, fundamentação e dispositivo pelas frases fixas; sem elas, devolve None e a sentença vai para o Jev."""
    disp = [p["n"] for p in pars if FRASE_DISPOSITIVO.match(p["texto"])]
    if not disp:
        return None
    fund = [p["n"] for p in pars if p["n"] < disp[-1] and FRASE_FUNDAMENTACAO.match(p["texto"])]
    if not fund:
        return None
    f, d = fund[-1], disp[-1]
    return {"relatorio": [p for p in pars if p["n"] <= f],
            "fundamentacao": [p for p in pars if f < p["n"] < d],
            "dispositivo": [p for p in pars if p["n"] >= d]}


separadas = [s for s in SENTENCAS if separar_partes(paragrafos(s["texto"]))]
sem_frases = [s for s in SENTENCAS if not separar_partes(paragrafos(s["texto"]))]
homologacoes = sum("HOMOLOGO" in s["texto"].upper() for s in sem_frases)
print(f"as frases fixas separam as três partes em {len(separadas)} de {len(SENTENCAS)} sentenças")
print(f"nas outras {len(sem_frases)}, {homologacoes} são homologações (sem argumento do juiz); as demais vão para o Jev (seção 2)")

as frases fixas separam as três partes em 166 de 226 sentenças
nas outras 60, 38 são homologações (sem argumento do juiz); as demais vão para o Jev (seção 2)


## 1. Montagem da taxonomia

**Passo 1.** Os parágrafos da fundamentação das sentenças já na plataforma, sem as citações, viram embeddings e são agrupados. Antes, nomes, valores, datas e empresas são mascarados, para que o agrupamento siga o argumento do juiz, e não os fatos. Um LLM lê exemplos de cada grupo e escreve o nome e a descrição da categoria.

Abaixo roda o preparo desses parágrafos nas sentenças reais. Os embeddings, o agrupamento e o LLM não rodam aqui.

In [3]:
ASPAS = re.compile(r"[\"“]([^\"”]{30,})[\"”]")
EMENTA = re.compile(r"(RECURSO (DES)?PROVIDO|Sentença mantida|\((TJSP|STJ|STF)[;,])")
VALOR = re.compile(r"R\$\s*[\d.]+,\d{2}(\s*\([^)]*\))?")
DATA = re.compile(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b|\b\d{1,2} de [a-zç]+ de \d{4}\b")
NUMERO_PROCESSO = re.compile(r"\b\d{7}-\d{2}\.\d{4}\.\d\.\d{2}\.\d{4}\b")
CNPJ = re.compile(r"\(?CNPJ[^)]*?\d{2}\.?\d{3}\.?\d{3}/\d{4}-\d{2}\)?")
EMPRESA = re.compile(r"\b([A-ZÁ-Ú][\wÀ-ú&.,\-]*\s+((e|de|do|da|dos|das)\s+)?){1,7}(S[./]?A\.?|[Ll][Tt][Dd][Aa]\.?)(?=[\s,.;)]|$)")


def marcar_citacoes(pars):
    """Citação: ementa transcrita (parágrafo inteiro) ou trecho entre aspas. Não conta como argumento do juiz."""
    for p in pars:
        p["ementa"] = bool(EMENTA.search(p["texto"]))
        p["citacoes"] = ASPAS.findall(p["texto"])
    return pars


def partes_do_cabecalho(texto):
    autor = re.search(r"Requerente:\s*(.+)", texto)
    reu = re.search(r"Requerido:\s*(.+)", texto)
    return (autor.group(1).strip() if autor else ""), (reu.group(1).strip() if reu else "")


def mascarar(texto, autor, reu):
    """Tira os fatos do caso (nomes, valores, datas, empresas, número do processo) antes dos embeddings."""
    texto = NUMERO_PROCESSO.sub("[PROCESSO]", DATA.sub("[DATA]", VALOR.sub("[VALOR]", texto)))
    texto = EMPRESA.sub("[EMPRESA]", CNPJ.sub("[CNPJ]", texto))
    if autor:
        texto = re.sub(rf"\b{re.escape(autor.split()[0])}(\s+[A-ZÁ-Ú][a-zá-ú]+)*", "[PARTE AUTORA]", texto)
    if reu:
        nome = re.sub(r"(?i)(\s+(S[./]?A\.?|Ltda\.?|Epp|ME))*(\s+e outros?)?\s*$", "", reu)
        for parte in (nome, " ".join(nome.split()[:2])):   # o nome inteiro e as duas primeiras palavras
            texto = re.sub(rf"(?i)\b{re.escape(parte)}(\s+(S[./]?A\.?|Ltda\.?))?(?=[\s,.;)]|$)", "[EMPRESA RÉ]", texto)
    return texto


def para_agrupar(sentenca):
    """Passo 1 da montagem, antes dos embeddings: só a fundamentação, sem citações, mascarada."""
    pars = marcar_citacoes(paragrafos(sentenca["texto"]))
    partes = separar_partes(pars)
    if partes is None:
        return []
    autor, reu = partes_do_cabecalho(sentenca["texto"])
    return [mascarar(ASPAS.sub("[CITAÇÃO]", p["texto"]), autor, reu) for p in partes["fundamentacao"] if not p["ementa"]]


mesmo_tipo = [s for s in SENTENCAS if tipo_de_processo(s) == tipo_de_processo(EXEMPLO)]
com_mascara = [t for s in mesmo_tipo for t in para_agrupar(s) if re.search(r"\[(PARTE AUTORA|EMPRESA RÉ|VALOR)\]", t)]
for t in com_mascara[:3]:
    print("-", t[:300], "\n")
print(f"parágrafos prontos para agrupar no tipo do exemplo: {sum(len(para_agrupar(s)) for s in mesmo_tipo)}, de {len(mesmo_tipo)} sentenças")

- Assim, comprovada a inadimplência, ao menos neste ponto, a inclusão da dívida de [VALOR] na coluna “prejuízo” do SCR/BACEN (fls. 98) é legítima. 

- A corré [EMPRESA RÉ], ao seu turno, relatou ter apresentado ao banco a alteração de contrato, informando-o a respeito da exclusão do requerente do quadro societário, mas não comprovou tal alegação (fls. 77/79). 

- Em consideração a estes critérios, faz-se suficiente para a compensação dos prejuízos extrapatrimoniais, em observância aos princípios da proporcionalidade e razoabilidade, a sua fixação em [VALOR]. 

parágrafos prontos para agrupar no tipo do exemplo: 46, de 11 sentenças


**Passo 2.** Advogados revisam a categoria dos parágrafos de um lote de sentenças e corrigem a lista; esse lote é o **gabarito de ajuste**. **Passo 3.** Teste: o Jev classifica o gabarito usando as descrições como opções; a categoria que erra muito tem a descrição reescrita pelos advogados.

A taxonomia abaixo é um **exemplo escrito à mão**, no formato que sai desses passos, para o tipo da sentença 61, só para as etapas seguintes terem opções. Argumentos comuns a qualquer processo ficam na taxonomia geral, usada junto com a do tipo; um tipo com poucas sentenças usa só a geral. A lista de pedidos de cada tipo é montada do mesmo jeito.

In [4]:
TAXONOMIA_GERAL = {   # exemplo
    "incompetencia_territorial": ("Incompetência territorial", "O juízo não é competente pelo domicílio das partes, e o processo é extinto."),
    "prescricao": ("Prescrição", "A pretensão está prescrita pelo decurso do prazo legal."),
}
TAXONOMIA_POR_TIPO = {tipo_de_processo(EXEMPLO): {   # exemplo
    "scr_bacen_nao_restritivo": ("SCR não é cadastro restritivo",
                                 "O registro no SCR do Banco Central é administrativo e não se equipara a cadastro restritivo de crédito."),
    "notificacao_previa_art43": ("Notificação prévia da inscrição",
                                 "Discute se a falta de comunicação prévia da inscrição (art. 43, § 2º, do CDC) a torna irregular."),
    "inversao_onus_prova_minima": ("Inversão do ônus e prova mínima",
                                   "A inversão do ônus da prova não dispensa o autor de provar minimamente os fatos do seu direito."),
    "dano_moral_nao_comprovado": ("Dano moral não comprovado", "Sem prova de prejuízo à personalidade, não há dano moral a indenizar."),
    "dano_moral_presumido_negativacao": ("Dano moral presumido na negativação indevida",
                                         "A inscrição indevida em cadastro de inadimplentes gera dano moral sem prova do abalo."),
}}
PEDIDOS_POR_TIPO = {tipo_de_processo(EXEMPLO): {   # exemplo; pedido = o que o autor pediu ao juiz
    "danos_morais": "indenização por danos morais",
    "exclusao_do_cadastro": "exclusão do nome ou do apontamento do cadastro",
}}


def categorias(tipo):
    """Taxonomia do tipo + taxonomia geral; um tipo sem taxonomia própria (poucas sentenças) usa só a geral."""
    return {**TAXONOMIA_POR_TIPO.get(tipo, {}), **TAXONOMIA_GERAL}


TIPO = tipo_de_processo(EXEMPLO)
CATEGORIAS, PEDIDOS = categorias(TIPO), PEDIDOS_POR_TIPO[TIPO]
print("categorias para o tipo do exemplo:", list(CATEGORIAS))
print("tipo sem taxonomia própria:", list(categorias((DADOS["classe"], "Bancários"))))

categorias para o tipo do exemplo: ['scr_bacen_nao_restritivo', 'notificacao_previa_art43', 'inversao_onus_prova_minima', 'dano_moral_nao_comprovado', 'dano_moral_presumido_negativacao', 'incompetencia_territorial', 'prescricao']
tipo sem taxonomia própria: ['incompetencia_territorial', 'prescricao']


## 2. Extração: preparação da sentença (código)

A sentença 61 é dividida em parágrafos numerados com a posição no texto, separada nas três partes pelas frases fixas, e as citações são marcadas. As súmulas, artigos e precedentes do parágrafo e das citações vizinhas vão para `fundamentos_citados`, por regra no código.

In [5]:
FUNDAMENTO = re.compile(r"(?i)\b(súmula\s+(nº\s*)?\d+(\s+do\s+(STJ|STF))?|(AgInt no )?REsp\s*[\d.]+(/[A-Z]{2})?"
                        r"|art(igo)?\.?\s*\d+[º°]?((,|\s)\s*(§\s*\d+º?|inc(iso|\.)?\s*[IVX]+|[IVX]+))*,?\s+d[oa]\s+"
                        r"(Código de Defesa do Consumidor|CDC|Código de Processo Civil|CPC|Lei( nº)?\s*[\d.]+(/\d+)?))")


TITULO = re.compile(r"(?i)^(d[oa]s?\s.{2,50}|preliminar(es)?|(do )?mérito)$")


def marcar_titulos(pars):
    """Título de seção do juiz ("Dos danos morais"): linha curta, sem pontuação final. Vai no contexto da etapa 2."""
    for p in pars:
        p["titulo"] = len(p["texto"]) <= 60 and not p["texto"].endswith((".", ":", ";", ",")) and bool(TITULO.match(p["texto"]))
    return pars


def fundamentos_citados(fund, nums):
    """Súmulas, artigos e precedentes dos parágrafos do argumento e das ementas vizinhas."""
    vizinhas = {n + d for n in nums for d in (-1, 1)}
    textos = [p["texto"] for p in fund if p["n"] in nums or (p["n"] in vizinhas and p["ementa"])]
    return sorted({re.sub(r"\s+", " ", m.group(0)) for t in textos for m in FUNDAMENTO.finditer(t)})


pars = marcar_titulos(marcar_citacoes(paragrafos(EXEMPLO["texto"])))
partes = separar_partes(pars)
for nome, lista in partes.items():
    print(f"{nome:<14} §{lista[0]['n']}–§{lista[-1]['n']}  começa em: {lista[0]['texto'][:60]}")
fund = partes["fundamentacao"]
print()
for p in fund:
    marca = "ementa (citação)" if p["ementa"] else (f"{len(p['citacoes'])} trecho(s) entre aspas" if p["citacoes"] else "")
    print(f"§{p['n']:<3}{p['texto'][:70]:<72}{marca}")
print("\nfundamentos citados em toda a fundamentação:", fundamentos_citados(fund, {p["n"] for p in fund}))
com_titulo = sum(any(p["titulo"] for p in marcar_titulos(paragrafos(s["texto"]))) for s in SENTENCAS)
print(f"sentenças reais com título de seção: {com_titulo} de {len(SENTENCAS)} (no Juizado os juízes não usam; o título vai vazio)")

relatorio      §1–§14  começa em: CONCLUSÃO
fundamentacao  §15–§26  começa em: No mérito, a ação é improcedente.
dispositivo    §27–§32  começa em: Ante o exposto, JULGO IMPROCEDENTE a ação e extinto o feito,

§15 No mérito, a ação é improcedente.                                       
§16 A relação estabelecida entre todas as partes configura-se como uma rel  
§17 Ainda assim, à requerente cabe evidenciar minimamente os fatos constit  1 trecho(s) entre aspas
§18 No caso vertente, verifica-se que o autor não questiona, de fato, a or  
§19 Com efeito, as informações constantes dos cadastros “SISBACEN-SCR” se   
§20 Isto é, cuida-se de um cadastro público, de consulta restrita, necessá  
§21 Desse modo, inaplicável ao caso o disposto no art. 43, § 2º, do Código  
§22 Cadastramento da informação, sem prévia notificação, que não caracteri  ementa (citação)
§23 Reitere-se que o requerente não nega a existência da relação jurídica   
§24 Dessarte, reconhecida a existência de relação jurídica

Quando as frases fixas não aparecem, o Jev classifica cada parágrafo como relatório, fundamentação ou dispositivo, e só o que fica abaixo da probabilidade mínima vai para revisão humana. A pergunta, para um parágrafo de uma sentença real sem as frases:

In [6]:
def pergunta_separar_partes(paragrafo):
    return {"estado": paragrafo["texto"],
            "perguntas": [{"id": "parte", "tipo": "escolha", "texto": "Este parágrafo da sentença é:",
                           "opcoes": ["relatório", "fundamentação", "dispositivo"]}]}


sem = next(s for s in sem_frases if "HOMOLOGO" not in s["texto"].upper() and len(s["texto"]) > 2000)
ps = paragrafos(sem["texto"])
print(json.dumps(pergunta_separar_partes(ps[len(ps) // 2]), ensure_ascii=False, indent=1))


def parte_ou_revisao(probs, minimo=0.85):   # probabilidade mínima de exemplo
    parte, p = max(probs.items(), key=lambda kv: kv[1])
    return parte if p >= minimo else "revisão humana"


print(parte_ou_revisao({"relatório": 0.05, "fundamentação": 0.92, "dispositivo": 0.03}), "|",
      parte_ou_revisao({"relatório": 0.48, "fundamentação": 0.47, "dispositivo": 0.05}), "  ← respostas de exemplo")

{
 "estado": "Dispensado o relatório, nos termos do artigo 38 da Lei nº 9.099/95.",
 "perguntas": [
  {
   "id": "parte",
   "tipo": "escolha",
   "texto": "Este parágrafo da sentença é:",
   "opcoes": [
    "relatório",
    "fundamentação",
    "dispositivo"
   ]
  }
 ]
}
fundamentação | revisão humana   ← respostas de exemplo


## 3. Etapa 1: pedidos (Jev)

O dispositivo é onde o juiz decide cada pedido. Para cada pedido da lista, o Jev responde se o dispositivo o decide e com qual resultado; uma pergunta a mais, de sim ou não, verifica se ele decide algum pedido fora da lista (se sim, a sentença vai para revisão). Todas vão na mesma chamada.

In [7]:
RESULTADOS = ["não decide", "procedente", "parcialmente procedente", "improcedente", "extinto sem mérito"]


def perguntas_etapa1(dispositivo, pedidos):
    perguntas = [{"id": f"resultado:{pid}", "tipo": "escolha", "texto": f"O dispositivo decide o pedido de {nome}? Com qual resultado?",
                  "opcoes": RESULTADOS} for pid, nome in pedidos.items()]
    perguntas.append({"id": "pedido_fora_da_lista", "tipo": "sim_ou_nao",
                      "texto": "O dispositivo decide algum pedido que não está nesta lista: " + "; ".join(pedidos.values()) + "?"})
    return {"estado": "\n".join(p["texto"] for p in dispositivo), "perguntas": perguntas}


req = perguntas_etapa1(partes["dispositivo"], PEDIDOS)
req["estado"] = req["estado"][:160] + "…"   # só para encurtar a saída; a chamada leva o dispositivo inteiro
print(json.dumps(req, ensure_ascii=False, indent=1))

{
 "estado": "Ante o exposto, JULGO IMPROCEDENTE a ação e extinto o feito, com resolução do mérito, nos termos do art. 487, I, do Código de Processo Civil.\nNos termos dos art…",
 "perguntas": [
  {
   "id": "resultado:danos_morais",
   "tipo": "escolha",
   "texto": "O dispositivo decide o pedido de indenização por danos morais? Com qual resultado?",
   "opcoes": [
    "não decide",
    "procedente",
    "parcialmente procedente",
    "improcedente",
    "extinto sem mérito"
   ]
  },
  {
   "id": "resultado:exclusao_do_cadastro",
   "tipo": "escolha",
   "texto": "O dispositivo decide o pedido de exclusão do nome ou do apontamento do cadastro? Com qual resultado?",
   "opcoes": [
    "não decide",
    "procedente",
    "parcialmente procedente",
    "improcedente",
    "extinto sem mérito"
   ]
  },
  {
   "id": "pedido_fora_da_lista",
   "tipo": "sim_ou_nao",
   "texto": "O dispositivo decide algum pedido que não está nesta lista: indenização por danos morais; exclusão do nome ou do

## 4. Etapa 2: argumentos (Jev), um parágrafo por vez

Cada chamada leva o parágrafo, o anterior e o título da seção (nesta sentença o juiz não usa títulos, então ele vai vazio). Na primeira chamada, o Jev responde se o parágrafo é argumento do juiz e, para cada categoria, se o parágrafo traz aquele argumento (sim ou não), então um parágrafo pode ter mais de um. Na segunda, para cada argumento achado: qual pedido ele sustenta (ou nenhum, se for preliminar), se é razão de decidir ou comentário de passagem e se favorece o autor ou o réu. O relatório e as citações não são classificados.

In [8]:
def titulo_da_secao(fund, i):
    return next((p["texto"] for p in reversed(fund[:i]) if p.get("titulo")), None)


def estado(fund, i):
    """Cada chamada leva só o parágrafo, o anterior e o título da seção."""
    return {"titulo_da_secao": titulo_da_secao(fund, i), "paragrafo_anterior": fund[i - 1]["texto"] if i else None,
            "paragrafo": fund[i]["texto"]}


def perguntas_chamada1(est, cats):
    perguntas = [{"id": "e_argumento", "tipo": "sim_ou_nao", "texto": "Este parágrafo é argumento do próprio juiz?"}]
    perguntas += [{"id": f"categoria:{cid}", "tipo": "sim_ou_nao", "texto": f"O parágrafo traz este argumento? {titulo}: {desc}"}
                  for cid, (titulo, desc) in cats.items()]
    return {"estado": est, "perguntas": perguntas}


def perguntas_chamada2(est, cid, pedidos):
    return {"estado": est, "perguntas": [
        {"id": "pedido", "tipo": "escolha", "texto": f"Qual pedido o argumento '{CATEGORIAS[cid][0]}' sustenta?",
         "opcoes": list(pedidos) + ["nenhum (preliminar)"]},
        {"id": "papel", "tipo": "escolha", "texto": "Esse argumento é:", "opcoes": ["razão de decidir", "comentário de passagem"]},
        {"id": "favorece", "tipo": "escolha", "texto": "Esse argumento favorece:", "opcoes": ["autor", "réu"]}]}


i = next(k for k, p in enumerate(fund) if p["texto"].startswith("Isto é, cuida-se"))
est = estado(fund, i)
print("estado:", {k: (v[:70] + "…" if v else v) for k, v in est.items()})
for q in perguntas_chamada1(est, CATEGORIAS)["perguntas"]:
    print(" 1ª chamada:", q["texto"][:110])
for q in perguntas_chamada2(est, "scr_bacen_nao_restritivo", PEDIDOS)["perguntas"]:
    print(" 2ª chamada:", q["texto"], q["opcoes"])

estado: {'titulo_da_secao': None, 'paragrafo_anterior': 'Com efeito, as informações constantes dos cadastros “SISBACEN-SCR” se …', 'paragrafo': 'Isto é, cuida-se de um cadastro público, de consulta restrita, necessá…'}
 1ª chamada: Este parágrafo é argumento do próprio juiz?
 1ª chamada: O parágrafo traz este argumento? SCR não é cadastro restritivo: O registro no SCR do Banco Central é administr
 1ª chamada: O parágrafo traz este argumento? Notificação prévia da inscrição: Discute se a falta de comunicação prévia da 
 1ª chamada: O parágrafo traz este argumento? Inversão do ônus e prova mínima: A inversão do ônus da prova não dispensa o a
 1ª chamada: O parágrafo traz este argumento? Dano moral não comprovado: Sem prova de prejuízo à personalidade, não há dano
 1ª chamada: O parágrafo traz este argumento? Dano moral presumido na negativação indevida: A inscrição indevida em cadastr
 1ª chamada: O parágrafo traz este argumento? Incompetência territorial: O juízo não é competente pelo d

**Sentenças longas.** A sentença nunca vai inteira ao modelo: uma sentença longa só gera mais chamadas, todas do mesmo tamanho. Um argumento espalhado em vários parágrafos é remontado na etapa 3 (na seção 5, os §19 e §20 viram um argumento só).

In [9]:
maior = max(separadas, key=lambda s: len(s["texto"]))
fund_m = separar_partes(paragrafos(maior["texto"]))["fundamentacao"]
maior_estado = max(len(json.dumps(estado(fund_m, k), ensure_ascii=False)) for k in range(len(fund_m)))
print(f"maior sentença: {len(maior['texto'])} caracteres → {len(fund_m)} chamadas da etapa 2, a maior com {maior_estado} caracteres de estado")

maior sentença: 19256 caracteres → 34 chamadas da etapa 2, a maior com 1837 caracteres de estado


## 5. Etapas 3 e 4: juntar e decidir

As respostas do Jev abaixo são **exemplos escritos à mão**, no formato que ele devolve (a probabilidade de cada resposta), para os parágrafos reais da sentença 61. As probabilidades mínimas também são **exemplo**: as reais são ajustadas só no gabarito de ajuste, de novo a cada versão, e a meta (a concordância entre dois advogados) vem do gabarito de avaliação. Escolhemos os menores valores com que os argumentos publicados ainda alcançam essa meta, e o acerto é medido no argumento inteiro, e não em cada resposta: por isso o argumento só é publicado se as cinco respostas passam.

In [10]:
def par(inicio):
    return next(p for p in fund if p["texto"].startswith(inicio))


RESP_ETAPA1 = {"danos_morais": ("improcedente", 0.97), "exclusao_do_cadastro": ("improcedente", 0.95), "pedido_fora_da_lista": 0.08}
resp = lambda cat, pedido, papel, lado: {"categoria": cat, "pedido": pedido, "papel": papel, "favorece": lado}
RESP_ETAPA2 = [   # (parágrafo, P(é argumento do juiz), {categoria com "sim": respostas}) — exemplo; as omitidas ficaram perto de zero
    (par("No mérito, a ação"), 0.05, {}),
    (par("A relação estabelecida"), 0.94,     # a única categoria com "sim" fica abaixo do mínimo → "outro"
     {"inversao_onus_prova_minima": resp(0.40, ("danos_morais", 0.50), ("razão de decidir", 0.70), ("réu", 0.80))}),
    (par("Ainda assim, à requerente"), 0.96,
     {"inversao_onus_prova_minima": resp(0.91, ("danos_morais", 0.58), ("razão de decidir", 0.88), ("réu", 0.95))}),
    (par("Com efeito, as informações"), 0.98,
     {"scr_bacen_nao_restritivo": resp(0.95, ("exclusao_do_cadastro", 0.86), ("razão de decidir", 0.93), ("réu", 0.97))}),
    (par("Isto é, cuida-se"), 0.97,
     {"scr_bacen_nao_restritivo": resp(0.90, ("exclusao_do_cadastro", 0.84), ("razão de decidir", 0.91), ("réu", 0.96)),
      "notificacao_previa_art43": resp(0.88, ("exclusao_do_cadastro", 0.81), ("razão de decidir", 0.90), ("réu", 0.94))}),
    (par("Frise-se, ainda"), 0.97,
     {"dano_moral_nao_comprovado": resp(0.94, ("danos_morais", 0.97), ("razão de decidir", 0.92), ("réu", 0.97))}),
]
PROB_MINIMA = {"argumento": 0.90, "categoria": 0.85, "pedido": 0.80, "papel": 0.80, "favorece": 0.80}   # exemplo, uma por pergunta
PROB_MINIMA_POR_CATEGORIA = {"scr_bacen_nao_restritivo": 0.88}   # exemplo: uma categoria com mínimo próprio


def minimo(pergunta, categoria=None):
    if pergunta == "categoria":
        return PROB_MINIMA_POR_CATEGORIA.get(categoria, PROB_MINIMA["categoria"])
    return PROB_MINIMA[pergunta]
PROB_DESCARTE = 0.10                        # P(é argumento) abaixo disto: "não é argumento do juiz" com confiança
POUCOS_EXEMPLOS = {"notificacao_previa_art43"}   # exemplo: categoria com poucos exemplos no gabarito

In [11]:
def candidatos(resp_etapa2):
    """Um candidato por (parágrafo, categoria); "outro" quando é argumento e nenhuma categoria passa."""
    saida = []
    for p, p_arg, achados in resp_etapa2:
        passaram = {cid: r for cid, r in achados.items() if r["categoria"] >= minimo("categoria", cid)}
        if p_arg < PROB_DESCARTE:
            saida.append({"paragrafos": [p], "categoria": None, "p_arg": p_arg})
            continue
        if not passaram:
            saida.append({"paragrafos": [p], "categoria": "outro", "p_arg": p_arg})
        for cid, r in passaram.items():
            saida.append({"paragrafos": [p], "categoria": cid, "pedido": r["pedido"][0], "papel": r["papel"][0], "favorece": r["favorece"][0],
                          "confianca": {"argumento": p_arg, "categoria": r["categoria"], "pedido": r["pedido"][1],
                                        "papel": r["papel"][1], "favorece": r["favorece"][1]}})
    return saida


def juntar(cands):
    """Etapa 3: mesma categoria e mesmo pedido em parágrafos vizinhos viram um argumento, com a menor confiança entre eles."""
    juntos = []
    for c in sorted(cands, key=lambda c: c["paragrafos"][0]["n"]):
        ant = next((j for j in juntos if j.get("pedido") and j["categoria"] == c["categoria"] and j.get("pedido") == c.get("pedido")
                    and j["paragrafos"][-1]["n"] + 1 == c["paragrafos"][0]["n"]), None)
        if ant:
            ant["paragrafos"] += c["paragrafos"]
            ant["confianca"] = {k: min(v, c["confianca"][k]) for k, v in ant["confianca"].items()}
        else:
            juntos.append(c)
    return juntos


def lado_bate(arg, resultados):
    r = resultados.get(arg.get("pedido"), (None,))[0]
    if r not in ("procedente", "improcedente"):
        return True
    return (arg["favorece"] == "autor") == (r == "procedente")


def amostra_fixa(arg):   # uma fração fixa dos descartados vai para revisão
    return int(hashlib.sha256(str(arg["paragrafos"][0]["n"]).encode()).hexdigest(), 16) % 10 == 0


def decidir(arg, resultados):
    """Etapa 4."""
    if arg["categoria"] is None:
        return "revisão (amostra dos descartados)" if amostra_fixa(arg) else "descartado"
    if arg["categoria"] == "outro":
        return "revisão (outro)"
    if not lado_bate(arg, resultados):
        return "revisão (lado ≠ resultado)"
    if arg["categoria"] in POUCOS_EXEMPLOS:
        return "revisão (categoria com poucos exemplos)"
    if all(arg["confianca"][k] >= minimo(k, arg["categoria"]) for k in PROB_MINIMA):
        return "publicado"
    return "revisão (abaixo da probabilidade mínima)"


resultados = {k: v for k, v in RESP_ETAPA1.items() if k != "pedido_fora_da_lista"}
print("pedido fora da lista?", "sim → revisão" if RESP_ETAPA1["pedido_fora_da_lista"] >= 0.5 else "não")
ARGS = juntar(candidatos(RESP_ETAPA2))
for a in ARGS:
    a["decisao"] = decidir(a, resultados)
    nums = "+".join(f"§{p['n']}" for p in a["paragrafos"])
    print(f"{nums:<10}{str(a['categoria']):<30}{a.get('pedido') or '':<22}{a['decisao']}")
virado = {**next(a for a in ARGS if a["categoria"] == "dano_moral_nao_comprovado"), "favorece": "autor"}
print("\nse o Jev dissesse que o §25 favorece o autor (pedido improcedente):", decidir(virado, resultados))

pedido fora da lista? não
§15       None                                                descartado
§16       outro                                               revisão (outro)
§17       inversao_onus_prova_minima    danos_morais          revisão (abaixo da probabilidade mínima)
§19+§20   scr_bacen_nao_restritivo      exclusao_do_cadastro  publicado
§20       notificacao_previa_art43      exclusao_do_cadastro  revisão (categoria com poucos exemplos)
§25       dano_moral_nao_comprovado     danos_morais          publicado

se o Jev dissesse que o §25 favorece o autor (pedido improcedente): revisão (lado ≠ resultado)


## 6. Saída estruturada e regra do JSON

Cada argumento é gravado no formato do `questao2.md`. O título e a descrição vêm da taxonomia e fazem o papel do resumo; o específico do caso fica em `fundamentos_citados`. O trecho é o texto do próprio parágrafo, recortado da sentença, nunca reescrito. `cliente_id` e `caso_id` não existem no dado público e ficam vazios; página e retângulos só existem com o PDF.

In [12]:
VERSOES = {"schema": "1.2", "taxonomia": "neg_v3", "classificador": "jev-2026-09", "limiares": "neg_v3_l2", "ocr": "ocr-2026-08", "codigo": "c41a"}
dia, mes, ano = EXEMPLO["data"].split("/")


def gravar(arg):
    nums = [p["n"] for p in arg["paragrafos"]]
    titulo, descricao = CATEGORIAS[arg["categoria"]]
    pedido = None if arg["pedido"] == "nenhum (preliminar)" else arg["pedido"]
    return {
        "id": "arg_" + hashlib.sha256(f"{EXEMPLO['processo']}{arg['categoria']}{nums}".encode()).hexdigest()[:4],
        "cliente_id": None, "caso_id": None, "documento_id": EXEMPLO["processo"],
        "hash_documento": "sha256:" + hashlib.sha256(EXEMPLO["texto"].encode()).hexdigest()[:12],
        "juiz": EXEMPLO["magistrada"], "vara": EXEMPLO["vara"], "data_sentenca": f"{ano}-{mes}-{dia}",
        "tipo_processo": TIPO[1], "categoria": arg["categoria"], "titulo": titulo, "descricao": descricao,
        "pedido": pedido, "resultado_pedido": resultados[pedido][0] if pedido else None,
        "papel": {"razão de decidir": "razao_de_decidir", "comentário de passagem": "comentario_de_passagem"}[arg["papel"]], "favorece": arg["favorece"],
        "fundamentos_citados": fundamentos_citados(fund, set(nums)),
        "trechos": [{"paragrafo": p["n"], "pagina": None, "char_ini": p["char_ini"], "char_fim": p["char_fim"], "retangulos": [],
                     "texto": EXEMPLO["texto"][p["char_ini"]:p["char_fim"]].strip()} for p in arg["paragrafos"]],
        "confianca": arg["confianca"],
        "status": "publicado" if arg["decisao"] == "publicado" else "em revisão", "origem": "jev",
        "versoes": VERSOES,
    }


def conferir_regra(g):
    """Regra do JSON: valores fechados, preliminar com pedido null, trecho igual ao texto do parágrafo."""
    assert g["categoria"] in CATEGORIAS
    assert g["pedido"] in PEDIDOS or (g["pedido"] is None and g["resultado_pedido"] is None)
    assert g["papel"] in ("razao_de_decidir", "comentario_de_passagem") and g["favorece"] in ("autor", "réu")
    assert all(t["texto"] == p["texto"] for t, p in zip(g["trechos"], next(a["paragrafos"] for a in ARGS if a.get("categoria") == g["categoria"])))
    return True


GRAVADOS = [gravar(a) for a in ARGS if a["categoria"] not in (None, "outro")]
print(all(conferir_regra(g) for g in GRAVADOS), "→ todos seguem a regra do JSON")
print(json.dumps(next(g for g in GRAVADOS if g["categoria"] == "inversao_onus_prova_minima"), ensure_ascii=False, indent=1))
prelim = gravar({**next(a for a in ARGS if a["categoria"] == "dano_moral_nao_comprovado"), "pedido": "nenhum (preliminar)"})
print("se a resposta fosse 'nenhum (preliminar)' → pedido:", prelim["pedido"], "| resultado_pedido:", prelim["resultado_pedido"],
      "| segue a regra:", conferir_regra(prelim))

True → todos seguem a regra do JSON
{
 "id": "arg_65e3",
 "cliente_id": null,
 "caso_id": null,
 "documento_id": "4369884-51.2024.8.26.1598",
 "hash_documento": "sha256:4050b8d8d17d",
 "juiz": "MAGISTRADA A",
 "vara": "1ª Vara do Juizado Especial Cível",
 "data_sentenca": "2024-06-19",
 "tipo_processo": "Inclusão Indevida em Cadastro de Inadimplentes",
 "categoria": "inversao_onus_prova_minima",
 "titulo": "Inversão do ônus e prova mínima",
 "descricao": "A inversão do ônus da prova não dispensa o autor de provar minimamente os fatos do seu direito.",
 "pedido": "danos_morais",
 "resultado_pedido": "improcedente",
 "papel": "razao_de_decidir",
 "favorece": "réu",
 "fundamentos_citados": [
  "AgInt no REsp 1717781/RO",
  "art. 373, inc. I, do Código de Processo Civil"
 ],
 "trechos": [
  {
   "paragrafo": 17,
   "pagina": null,
   "char_ini": 2065,
   "char_fim": 2587,
   "retangulos": [],
   "texto": "Ainda assim, à requerente cabe evidenciar minimamente os fatos constitutivos do seu d

O tipo do argumento é a categoria. Os principais argumentos são os de papel "razão de decidir". O hash do documento evita processar a mesma sentença duas vezes:

In [13]:
JA_PROCESSADAS = {g["hash_documento"] for g in GRAVADOS}
def processar_se_nova(sentenca):
    h = "sha256:" + hashlib.sha256(sentenca["texto"].encode()).hexdigest()[:12]
    return "já processada, ignorada" if h in JA_PROCESSADAS else "nova, segue para a extração"
print("sentença 61 de novo:", processar_se_nova(EXEMPLO), "| sentença 35:", processar_se_nova(SENTENCAS[35]))
print("principais (razão de decidir):", [g["categoria"] for g in GRAVADOS if g["papel"] == "razao_de_decidir"])

sentença 61 de novo: já processada, ignorada | sentença 35: nova, segue para a extração
principais (razão de decidir): ['inversao_onus_prova_minima', 'scr_bacen_nao_restritivo', 'notificacao_previa_art43', 'dano_moral_nao_comprovado']


## 7. Revisão humana

O revisor vê o trecho destacado no PDF e aceita, corrige ou rejeita o argumento, ou marca um trecho que faltou. Cada ação é um registro novo, sem apagar o original, e muda o status e a origem. Os "outro" acumulados voltam aos passos 1 e 2 da montagem.

In [14]:
EVENTOS = []   # só inserção: o original nunca é alterado
STATUS_DA_ACAO = {"aceita": "publicado", "corrige": "corrigido", "rejeita": "rejeitado", "marca_trecho": "em revisão"}


def revisar(arg_id, acao, revisor, **mudancas):
    EVENTOS.append({"argumento": arg_id, "acao": acao, "revisor": revisor, "mudancas": mudancas,
                    "status": STATUS_DA_ACAO[acao], "origem": "advogado"})


def atual(g):
    """O argumento como a base o mostra: o original mais os eventos de revisão."""
    v = dict(g)
    for e in (e for e in EVENTOS if e["argumento"] == g["id"]):
        v.update(e["mudancas"], status=e["status"], origem=e["origem"])
    return v


inv = next(g for g in GRAVADOS if g["categoria"] == "inversao_onus_prova_minima")
revisar(inv["id"], "corrige", "adv_3", pedido="exclusao_do_cadastro", resultado_pedido="improcedente")
print("original:", inv["pedido"], inv["status"], inv["origem"])
print("atual:   ", atual(inv)["pedido"], atual(inv)["status"], atual(inv)["origem"])
OUTROS = [a["paragrafos"][0]["texto"][:60] for a in ARGS if a["categoria"] == "outro"]
print("\n\"outro\" guardados para voltar à montagem:", OUTROS)

original: danos_morais em revisão jev
atual:    exclusao_do_cadastro corrigido advogado

"outro" guardados para voltar à montagem: ['A relação estabelecida entre todas as partes configura-se co']


## 8. Qualidade

Dois advogados marcam os argumentos de sentenças sorteadas sem ver o sistema, e um terceiro desempata: é o **gabarito de avaliação**. Todo mês em produção medimos a precisão e o recall por categoria, se os publicados acertam tanto quanto os advogados concordam entre si, quanto vai para revisão e os erros na separação das partes. A regra de acerto (o argumento do gabarito abaixo é um exemplo):

In [15]:
def acerta(sistema, gabarito):
    """Mesma categoria e mesmo pedido de um argumento do gabarito, com pelo menos um parágrafo em comum."""
    return (sistema["categoria"] == gabarito["categoria"] and sistema["pedido"] == gabarito["pedido"]
            and bool({t["paragrafo"] for t in sistema["trechos"]} & set(gabarito["paragrafos"])))


scr = next(g for g in GRAVADOS if g["categoria"] == "scr_bacen_nao_restritivo")
nums_scr = [t["paragrafo"] for t in scr["trechos"]]
print("parágrafos do argumento do sistema:", nums_scr)
print("acerta:", acerta(scr, {"categoria": "scr_bacen_nao_restritivo", "pedido": "exclusao_do_cadastro", "paragrafos": [nums_scr[-1]]}))
print("acerta:", acerta(scr, {"categoria": "scr_bacen_nao_restritivo", "pedido": "danos_morais", "paragrafos": [nums_scr[-1]]}))

parágrafos do argumento do sistema: [19, 20]
acerta: True
acerta: False


## 9. Versionamento

Cada argumento grava a versão do schema, da taxonomia (as descrições definem as perguntas do Jev e fazem o papel do prompt), do classificador, das probabilidades mínimas, do OCR e do código. Nada é apagado, e a correção de um advogado prevalece sobre a reclassificação.

In [16]:
def o_que_reprocessar(mudanca, categorias_afetadas=()):
    return {
        "taxonomia": f"reclassifica só as categorias afetadas: {list(categorias_afetadas)}",
        "classificador": "reclassifica tudo, desde que a versão nova não fique pior no gabarito",
        "llm_ou_prompt_da_montagem": "gera uma versão nova da taxonomia, que passa de novo pelos advogados",
        "schema_campo_novo": "o campo fica vazio nos registros antigos",
        "schema_campo_muda_de_sentido": "migração dos registros antigos",
        "ocr_ou_divisao_em_paragrafos": "reprocessa a sentença: a posição dos trechos muda",
    }[mudanca]


def reclassificar(g, nova_taxonomia, mapa_categorias, resposta_nova):
    """A reclassificação gera um registro novo (o antigo fica); a categoria antiga é mapeada para a nova,
    e as correções dos advogados, ligadas ao argumento, continuam valendo por cima."""
    novo = {**g, **resposta_nova, "categoria": mapa_categorias.get(g["categoria"], g["categoria"]),
            "versoes": {**g["versoes"], "taxonomia": nova_taxonomia}}
    return novo


for m in ("taxonomia", "classificador", "llm_ou_prompt_da_montagem", "schema_campo_novo", "schema_campo_muda_de_sentido",
          "ocr_ou_divisao_em_paragrafos"):
    print(f"{m:<30} → {o_que_reprocessar(m, ['notificacao_previa_art43'])}")
print("\nversões gravadas no argumento:", GRAVADOS[0]["versoes"])
novo = reclassificar(inv, "neg_v4", {"inversao_onus_prova_minima": "onus_da_prova"}, {"pedido": "danos_morais"})
print("\nreclassificado pela neg_v4:", novo["categoria"], "| o Jev diz de novo:", novo["pedido"],
      "| a base mostra:", atual(novo)["pedido"], f"({atual(novo)['origem']}: a correção prevalece)")
print("registro antigo continua gravado:", inv["versoes"]["taxonomia"], inv["categoria"])

taxonomia                      → reclassifica só as categorias afetadas: ['notificacao_previa_art43']
classificador                  → reclassifica tudo, desde que a versão nova não fique pior no gabarito
llm_ou_prompt_da_montagem      → gera uma versão nova da taxonomia, que passa de novo pelos advogados
schema_campo_novo              → o campo fica vazio nos registros antigos
schema_campo_muda_de_sentido   → migração dos registros antigos
ocr_ou_divisao_em_paragrafos   → reprocessa a sentença: a posição dos trechos muda

versões gravadas no argumento: {'schema': '1.2', 'taxonomia': 'neg_v3', 'classificador': 'jev-2026-09', 'limiares': 'neg_v3_l2', 'ocr': 'ocr-2026-08', 'codigo': 'c41a'}

reclassificado pela neg_v4: onus_da_prova | o Jev diz de novo: danos_morais | a base mostra: exclusao_do_cadastro (advogado: a correção prevalece)
registro antigo continua gravado: neg_v3 inversao_onus_prova_minima


## 10. Argumentos inventados: o caminho com um LLM (plano B)

O LLM só escreve texto na montagem (nomes e descrições das categorias), e tudo passa pelos advogados (seção 1). Na extração nenhum modelo escreve texto: o Jev só escolhe opções e o trecho vem do documento (seção 6). Se o classificador for um LLM, as mesmas regras valem: ele responde só com números de parágrafo e opções da lista, em saída restrita; o código descarta o que não existe e recorta o trecho do próprio documento; e a confiança vem da concordância entre várias respostas dele à mesma pergunta. As respostas abaixo são **exemplo**.

In [17]:
def schema_restrito(fund, cats):
    """Saída restrita: só números de parágrafo e opções da lista, nunca texto livre."""
    return {"type": "object", "additionalProperties": False, "required": ["paragrafo", "categoria"],
            "properties": {"paragrafo": {"enum": [p["n"] for p in fund]}, "categoria": {"enum": list(cats) + ["outro"]}}}


def validar_resposta_llm(r, fund, cats):
    """Descarta número de parágrafo ou opção que não existe; o trecho é recortado do documento."""
    p = next((p for p in fund if p["n"] == r.get("paragrafo")), None)
    if p is None or r.get("categoria") not in list(cats) + ["outro"]:
        return None
    return {"categoria": r["categoria"], "paragrafo": p["n"], "trecho": EXEMPLO["texto"][p["char_ini"]:p["char_fim"]].strip()}


def confianca_por_concordancia(respostas):
    """O LLM não dá probabilidade confiável: a confiança é a fração de respostas iguais à mais comum."""
    validas = [r["categoria"] for r in respostas if r]
    mais_comum, n = Counter(validas).most_common(1)[0]
    return mais_comum, n / len(respostas)


n_frise = par("Frise-se, ainda")["n"]
RESPOSTAS_LLM = [{"paragrafo": n_frise, "categoria": "dano_moral_nao_comprovado"}] * 4 + [{"paragrafo": 999, "categoria": "dano_moral_nao_comprovado"}]
print("schema:", json.dumps(schema_restrito(fund, CATEGORIAS)["properties"]["categoria"], ensure_ascii=False)[:120], "…")
validadas = [validar_resposta_llm(r, fund, CATEGORIAS) for r in RESPOSTAS_LLM]
print("validadas:", [v and v["categoria"] for v in validadas], "(a resposta com o §999, que não existe, é descartada)")
print("categoria e confiança:", confianca_por_concordancia(validadas))

schema: {"enum": ["scr_bacen_nao_restritivo", "notificacao_previa_art43", "inversao_onus_prova_minima", "dano_moral_nao_comprova …
validadas: ['dano_moral_nao_comprovado', 'dano_moral_nao_comprovado', 'dano_moral_nao_comprovado', 'dano_moral_nao_comprovado', None] (a resposta com o §999, que não existe, é descartada)
categoria e confiança: ('dano_moral_nao_comprovado', 0.8)


## 11. Cobertura do texto e trade-offs

| Trecho do `questao2.md` | Onde está |
|---|---|
| Premissas 1 a 4, a solução | introdução e seção 0 |
| Montagem: passos 1 (preparo real), 2 e 3; taxonomia geral; lista de pedidos; tipo de processo | seção 1 |
| PDF/OCR; parágrafos; frases fixas e o Jev quando elas faltam (com a probabilidade mínima); citações; título da seção | seções 0 e 2 |
| Etapa 1 (incluindo pedido fora da lista) | seção 3 |
| Etapa 2 (duas chamadas, título da seção, mais de um argumento por parágrafo) e sentenças longas | seção 4 |
| Etapa 3 (junção, menor confiança, lado × resultado) | seção 5 |
| Etapa 4 (mínimo por pergunta e por categoria, "outro" pelo mínimo, descarte com amostra fixa, categoria rara, revisão) | seção 5 |
| Saída estruturada, regra do JSON (inclusive preliminar com null), tipo, resumo, fundamentos citados, principais, hash | seção 6 |
| Revisão humana (ações, registro novo, status, origem, "outro" de volta à montagem) | seção 7 |
| Qualidade (gabarito de avaliação, regra de acerto, medidas) | seção 8 |
| Versionamento (versões gravadas, casos 1 a 5, correção humana prevalece) | seção 9 |
| Argumentos inventados, com o Jev e com um LLM | seções 6 e 10 |

**Trade-offs** (do texto): a lista fechada deixa a base comparável e barata, mas um argumento novo fica como "outro" até a revisão; fundamentos diferentes na mesma categoria têm o mesmo resumo e só se distinguem pelo trecho e pelos fundamentos citados; a qualidade depende dos advogados; o Jev está em acesso antecipado, e o versionamento permite trocá-lo (seção 10); a sentença vai para um serviço externo, o que exige contrato de tratamento de dados com a TypeSafe, e processos em segredo de justiça não são enviados.